# JiT-S2-Spatial-Mamba — no gate, **d_state 4**, per-state SASF kernels

The no-gate Spatial-Mamba run (FID 93.95) used Spatial-Mamba's d_state 1. Going from d_state 16 to 1 cost VMamba +5.1 FID, so a larger state may help Spatial-Mamba too — but d_state 16 is slow here (≈36 min/epoch on a T4, ≈60 h for 100 epochs). This run is the middle point: **4 states per channel**, without activation checkpointing.

$$x_t[c,n] = e^{\Delta_t[c] A[c,n]}\,x_{t-1}[c,n] + \Delta_t[c]\,B_t[n]\,u_t[c],\qquad h = \mathrm{SASF}(x),\qquad y_t[c] = \sum_{n=1}^{4} C_t[n]\,h_t[c,n] + D[c]\,u_t[c]$$

- Same code path as the d_state 16 runs: the scan runs over 384 × 4 single-state channels on the **stock** kernel, which is the 4-state Mamba-1 recurrence exactly (`tests/test_spatial_dstate16_cpu.py` proves it for general N).
- SASF fuses all 1,536 state maps per block with **per-state kernels**: every (channel, state) map has its own dilated 3×3 kernels, $h_{c,n}=\sum_d \alpha_d\,\mathrm{DWConv}^{(c,n)}_d(x_{c,n})$ — +31,104 parameters per block vs shared kernels, same compute and memory.
- **No checkpointing:** ≈8.3 GiB of activations at batch 128 (bf16), est. peak ≈10–11 GiB, fits the 15 GB T4. d_state 16 needed ≈24 GiB and checkpointing.
- Everything else as the d_state 1 no-gate run: expand 1, **no gate**, LPU (zero-init), dilations [1, 2, 3], adaLN-Zero only.

Model: **30.10M** params.

| model (Tiny-ImageNet, 100 epochs) | params | FID-10K |
|---|---|---|
| JiM baseline: VMamba, d_state 16 | 31.03M | **91.73** |
| VMamba, d_state 1 | 30.20M | 96.86 |
| Spatial, gate, d_state 1 + LPU | 31.46M | 95.58 |
| Spatial, no gate, d_state 1 + LPU | 29.69M | 93.95 |
| Spatial, no gate, d_state 4, shared + LPU | 29.73M | ? |
| **this:** Spatial, no gate, d_state 4, per-state + LPU | 30.10M | ? |
| Spatial, no gate, d_state 16, shared + LPU | 29.89M | ? |

The verify cell prints this run's it/s and peak memory (section `4.` of the GPU check) — check the minutes per epoch before committing to 100 epochs.

---

JiT-S2 on Tiny-ImageNet-200 (64px, patch 8, 200 classes) via the repo's `run_experiment.py` / `evaluate.py`. Comment out the download cell if you attach a dataset instead, and the eval cell if you only want to train.

## 1. Environment  *(Internet ON)*

In [ ]:
import os, sys

# Pick the stack by the image's Python. torch 2.5.1 has no matching torchvision
# for Python 3.13 (Kaggle's newer image), so 3.13 moves to torch 2.6.0; 3.12
# keeps torch 2.5.1 exactly as every earlier run. mamba-ssm 2.2.4 and
# causal-conv1d 1.5.0.post8 publish cp312 and cp313 wheels for both.
PY = f"cp{sys.version_info.major}{sys.version_info.minor}"
if sys.version_info >= (3, 13):
    TORCH, TVISION, TAUDIO, TTAG = "2.6.0", "0.21.0", "2.6.0", "torch2.6"
else:
    TORCH, TVISION, TAUDIO, TTAG = "2.5.1", "0.20.1", "2.5.1", "torch2.5"
print("python", sys.version.split()[0], "->", PY, "| torch", TORCH)

# 1) Pin torch
!pip install -q torch=={TORCH} torchvision=={TVISION} torchaudio=={TAUDIO} \
    --index-url https://download.pytorch.org/whl/cu124

# 2) Download wheels with explicit destination
CAUSAL = f"causal_conv1d-1.5.0.post8+cu12{TTAG}cxx11abiFALSE-{PY}-{PY}-linux_x86_64.whl"
MAMBA  = f"mamba_ssm-2.2.4+cu12{TTAG}cxx11abiFALSE-{PY}-{PY}-linux_x86_64.whl"

os.system(f"wget -q https://github.com/Dao-AILab/causal-conv1d/releases/download/v1.5.0.post8/{CAUSAL} -O /kaggle/working/{CAUSAL}")
os.system(f"wget -q https://github.com/state-spaces/mamba/releases/download/v2.2.4/{MAMBA} -O /kaggle/working/{MAMBA}")
for w in (CAUSAL, MAMBA):
    p = f"/kaggle/working/{w}"
    assert os.path.exists(p) and os.path.getsize(p) > 0, f"wheel download failed: {w}"

!pip install -q /kaggle/working/{CAUSAL}
!pip install -q /kaggle/working/{MAMBA}

# 3) Patch mamba-ssm
import glob
for path in glob.glob("/usr/local/lib/python*/dist-packages/mamba_ssm/utils/generation.py"):
    with open(path) as f: src = f.read()
    new = src.replace(
        "from transformers.generation import GreedySearchDecoderOnlyOutput, SampleDecoderOnlyOutput, TextStreamer",
        "from transformers.generation import GenerateDecoderOnlyOutput, TextStreamer",
    ).replace(
        "output_cls = GreedySearchDecoderOnlyOutput if top_k == 1 else SampleDecoderOnlyOutput",
        "output_cls = GenerateDecoderOnlyOutput",
    )
    if new != src:
        with open(path, "w") as f: f.write(new)
        print(f"\u2705 Patched {path}")

print(">>> RESTART RUNTIME NOW <<<")

## 2. Repo  *(clone + cd)*

In [ ]:
# Clone the repo. The models, configs, wiring, CPU tests and the GPU check are on main.
import os
REPO_DIR = "/kaggle/working/thesis_Choustoulakis"
if not os.path.exists(REPO_DIR):
    !git clone -q https://github.com/Rodamanthosch/thesis_Choustoulakis.git {REPO_DIR}
else:
    !git -C {REPO_DIR} pull -q
%cd {REPO_DIR}
# sanity: the d_state-16 SASF option is wired into both entry points
!grep -q 'sasf_state_kernels' scripts/run_experiment.py && grep -q 'sasf_state_kernels' scripts/evaluate.py \
    && echo "wiring OK (train + eval)" || echo "WIRING MISSING -- pull latest main"

## 3. Verify the model  *(first session only; ~3-6 min)*

In [ ]:
# === Verify the model (~3-6 min; first session only -- comment out on resume).
# PYTHONPATH=. is REQUIRED: without it "from src.models..." fails and the
# tests' try/except misreports it as a missing mamba_ssm (known issue).
# If anything FAILS, STOP -- do not train a model that is not the one reported.
#  CPU proofs: the d_state 1 Spatial model is unchanged (all its checks), and at
#  d_state 16 the scan over 384 x 16 single-state channels IS the 16-state
#  recurrence, the mixer equals a naive per-token transcription (shared and
#  per-state kernels, float64), shared == per-state with tied kernels, param
#  counts, identity at init, and both build_models agree on the configs.
!PYTHONPATH=. python tests/test_spatial_mamba_cpu.py
!PYTHONPATH=. python tests/test_spatial_dstate16_cpu.py
#  GPU check (scripts/check_sasf_cuda.py): the stock CUDA scan returns the exact
#  N-state recurrence, every new mixer on CUDA matches the CPU reference, the
#  three configs train under bf16, and it/s + peak memory at batch 128 for the
#  new runs and their references -- note the it/s for the complexity table.
!PYTHONPATH=. python scripts/check_sasf_cuda.py

## 4. Download Tiny-ImageNet  *(to /tmp; comment out if you attach a dataset)*

In [ ]:
# Download Tiny-ImageNet-200 to /tmp (EPHEMERAL: keeps your saved Version small --
# only checkpoints go to /kaggle/working). Re-downloads each session (~2-3 min).
# Faster alternative: attach a Kaggle tiny-imagenet dataset and set DATA_DIR to it,
# then comment this cell out. Requires Internet ON.
import os, zipfile, glob, shutil
DATA_DIR = "/tmp/tiny-imagenet-200"
ZIP      = "/tmp/tiny-imagenet-200.zip"
SRC      = "http://cs231n.stanford.edu/tiny-imagenet-200.zip"

if not os.path.isdir(os.path.join(DATA_DIR, "train")):
    if not os.path.exists(ZIP):
        print("downloading tiny-imagenet-200 (~240MB)...")
        os.system(f"wget -q -O {ZIP} {SRC}")
    print("extracting...")
    with zipfile.ZipFile(ZIP) as z:
        z.extractall("/tmp")

# TRAIN: train/<cls>/images/*.JPEG -> train/<cls>/*.JPEG  (ImageFolder layout)
tr = os.path.join(DATA_DIR, "train")
for cls in os.listdir(tr):
    sub = os.path.join(tr, cls, "images")
    if os.path.isdir(sub):
        for f in glob.glob(os.path.join(sub, "*.JPEG")):
            shutil.move(f, os.path.join(tr, cls))
        shutil.rmtree(sub)
    for b in glob.glob(os.path.join(tr, cls, "*_boxes.txt")):
        os.remove(b)

# VAL: flat val/images + val_annotations.txt -> val/<cls>/*.JPEG  (evaluate.py reads val/)
val = os.path.join(DATA_DIR, "val")
ann = os.path.join(val, "val_annotations.txt")
if os.path.exists(ann):
    with open(ann) as f:
        for line in f:
            img, cls = line.split("\t")[:2]
            os.makedirs(os.path.join(val, cls), exist_ok=True)
            s = os.path.join(val, "images", img)
            if os.path.exists(s):
                shutil.move(s, os.path.join(val, cls, img))
    shutil.rmtree(os.path.join(val, "images"), ignore_errors=True)
    os.remove(ann)

print("train classes:", len(glob.glob(tr+"/*")),
      "| val classes:", len(glob.glob(val+"/*")), "| DATA_DIR:", DATA_DIR)

## 5. Settings  *(edits the cloned config; finds the checkpoint to resume)*

In [ ]:
# ── Settings for THIS notebook ─────────────────────────────────────────
EPOCHS     = 100          # full target; resume across sessions until reached
SAVE_FREQ  = 1            # overwrite checkpoint-last.pt EVERY epoch (12h safety)

# Fallback if the download cell was commented out (attached-dataset workflow):
try:
    DATA_DIR
except NameError:
    DATA_DIR = "/kaggle/input/tiny-imagenet/tiny-imagenet-200"  # <-- your attached dataset
    print("DATA_DIR not set by a download cell; using:", DATA_DIR)

CONFIG  = "configs/tiny_imagenet/jit-s2-spatial-nogate-dstate4-perstate.yaml"
RUN     = "exp_spatial_nogate_d4_perstate"
OUT_DIR = f"/kaggle/working/{RUN}"   # persists in the saved Version

# Sync the cloned config so BOTH training and evaluation read identical values
# (evaluate.py has no model CLI overrides -- it builds the model straight from
# config). Every conditioning arm stays OFF (adaLN-Zero baseline scaffold).
import re
def set_cfg(path, kv):
    s = open(path).read()
    for k, v in kv.items():
        s, n = re.subn(rf"(?m)^(\s*{k}:).*$", rf"\1 {v}", s, count=1)
        assert n == 1, f"key {k!r} not found in {path}"
    open(path, "w").write(s)
set_cfg(CONFIG, {"data_dir": DATA_DIR, "gate": "false", "d_state": 4, "expand": 1,
                 "sasf_dilations": "[1, 2, 3]", "lpu": "zero",
                 "sasf_state_kernels": "per_state",
                 "grad_checkpoint": "false", "ffn": "swiglu"})

# ── Resume across sessions AND accounts (automatic) ────────────────────
# Looks for THIS model's newest checkpoint in: this session's OUT_DIR, any
# attached notebook output (.../checkpoint-last.pt), and any attached DATASET
# holding an uploaded checkpoint (Kaggle unpacks an uploaded .pt into a folder
# with data.pkl / data/ / byteorder / version -- it is zipped back here).
# A checkpoint is used only if its weights match this config's model exactly
# (same names and shapes), so another run's checkpoint is never picked up.
# The newest epoch wins. To force one, set RESUME_CKPT_FORCE to its path.
RESUME_CKPT_FORCE = None
import os, glob, zipfile, torch, yaml
from scripts.evaluate import build_model
_ref = {k: tuple(v.shape) for k, v in build_model(yaml.safe_load(open(CONFIG))).state_dict().items()}

def _rebuild(folder):
    """Zip an unpacked checkpoint folder back into a loadable .pt in /tmp."""
    dst = os.path.join("/tmp/resume", folder.strip("/").replace("/", "__") + ".pt")
    if not os.path.exists(dst):
        os.makedirs("/tmp/resume", exist_ok=True)
        with zipfile.ZipFile(dst, "w", zipfile.ZIP_STORED) as z:
            for root, _, files in os.walk(folder):
                for f in files:
                    full = os.path.join(root, f)
                    z.write(full, os.path.join("checkpoint", os.path.relpath(full, folder)))
    return dst

cands = [os.path.join(OUT_DIR, "checkpoint-last.pt")]
cands += sorted(glob.glob("/kaggle/input/**/checkpoint-last.pt", recursive=True))
cands += [_rebuild(os.path.dirname(p))
          for p in sorted(glob.glob("/kaggle/input/**/data.pkl", recursive=True))]
best = None
print("checkpoints found:")
for c in ([RESUME_CKPT_FORCE] if RESUME_CKPT_FORCE else cands):
    if not (c and os.path.exists(c)):
        continue
    try:
        p = torch.load(c, map_location="cpu", weights_only=False)
    except Exception as e:
        print(f"  unreadable ({type(e).__name__}): {c}")
        continue
    match = {k: tuple(v.shape) for k, v in p.get("model", {}).items()} == _ref
    print(f"  {'THIS model ' if match else 'other model'}  epoch {p.get('epoch')}  {c}")
    if match and (best is None or p["epoch"] > best[0]):
        best = (p["epoch"], c)
    del p
RESUME_CKPT = best[1] if best else None

print("config :", CONFIG, "|", "spatial_mamba no gate, d_state 4, sasf_state_kernels=per_state, lpu zero, no checkpointing")
print("data   :", DATA_DIR, "(exists:", os.path.isdir(DATA_DIR), ")")
print("out    :", OUT_DIR)
print("resume :", f"{RESUME_CKPT} (last finished epoch {best[0]})" if best else "(fresh start)")

## 6. Train  *(Save Version before 12h; next session it resumes by itself)*

In [ ]:
# ── TRAIN ──────────────────────────────────────────────────────────────
# data_dir + the model knobs now live in the (edited) config; we only override run-specific
# knobs here. Runs in a subprocess (fresh torch, no restart).
CMD = (f"python scripts/run_experiment.py --config {CONFIG} "
       f"checkpoint.output_dir={OUT_DIR} checkpoint.save_last_freq={SAVE_FREQ} "
       f"training.epochs={EPOCHS}")
if RESUME_CKPT:
    CMD += f" checkpoint.resume_from={RESUME_CKPT}"
print(CMD, "\n" + "="*70)
!{CMD}

## 7. Evaluate  *(comment out the whole cell to skip)*

In [ ]:
import glob, os

# Find the checkpoint to evaluate: this session's OUT_DIR first, then any
# attached notebook output (Save Version -> attach it as an input). Searches
# instead of hardcoding a notebook hash, so nothing needs editing after the
# first Save Version.
cands = []
for root in (os.path.dirname(OUT_DIR), "/kaggle/input"):
    for which in ("checkpoint-best.pt", "checkpoint-last.pt"):
        cands += sorted(glob.glob(f"{root}/**/{RUN}/{which}", recursive=True))
assert cands, f"no checkpoint found for {RUN} -- train first, or attach the saved Version"
EVAL_CKPT = cands[0]
EVAL_OUT  = f"/kaggle/working/{RUN}/eval"
print("eval ckpt:", EVAL_CKPT)

EVAL_CMD = (f"python scripts/evaluate.py --config {CONFIG} --checkpoint {EVAL_CKPT} "
            f"--n_samples 10000 --batch_size 200 --ema 1 "
            f"--cfg_scale 2.5 --cfg_interval 0.1 1.0 --out_dir {EVAL_OUT}")
print(EVAL_CMD, "\n" + "="*70)
!PYTHONPATH=. {EVAL_CMD}

## Notes
- Checkpoints (`-last`/`-best`/`-ep###`) with both EMA copies + optimizer are in `OUT_DIR` under `/kaggle/working`, so a resumed run equals a continuous one.
- **Resuming, also on another Kaggle account:** attach the previous session's notebook output, or upload its `checkpoint-last.pt` as a dataset and attach that. The settings cell finds it, keeps only checkpoints of this exact model (d_state 4 checkpoints differ in shape from d_state 1 / 16 ones, so they are never mixed up), and resumes from the newest epoch. Check its `resume :` line before training.
- Keep `EPOCHS`, `cfg_scale`, `DATA_DIR`, seed and sampler identical to the other runs.
- Comparisons: **this vs d_state 4 shared** (sister notebook `jit-s2-spatial-tinyin-nogate-dstate4-shared.ipynb`) = whether per-state SASF kernels are worth their parameters at the same state size and compute; **this vs Spatial no gate d_state 1 (93.95)** = 1 → 4 states; **this vs the JiM baseline (91.73)** = Spatial-Mamba with a small state vs SS2D with d_state 16.
- **Memory / speed:** no activation checkpointing (fits at batch 128), so no recompute; the state traffic is ¼ of d_state 16's. If a T4 still runs out of memory (the GPU check reports `OUT OF MEMORY` or training crashes in the first step), set `"grad_checkpoint": "true"` in the settings cell -- same maths, less memory, slower.
- Data lives in `/tmp` (not saved in the Version) to keep commits small; it re-downloads each session. To skip that, attach a Kaggle tiny-imagenet dataset and set `DATA_DIR` to it.